# landsat-record examples

How to use the `landsat-record` package, from a terminal and from Python. It builds a browsable record of every
Landsat Collection 2 Level 1 scene over an Antarctic site:

- crops at native resolution;
- Cloud Optimized GeoTIFFs;
- a viewer page that can be published to a public bucket.

**Run this on CryoCloud**, which can read the requester-pays USGS Landsat bucket. The
[README](https://github.com/ldeo-glaciology/landsat-record#readme) covers every option.

## 1. Install, and store the key for publishing

Installing is needed once per CryoCloud account; `--upgrade` updates it.

In [ ]:
%pip install --user --quiet git+https://github.com/ldeo-glaciology/landsat-record

Publishing to `gs://ldeo-glaciology` needs the bucket's service-account key. Upload it to your home folder
(not inside a git repository), then store it once:

- the key is copied to `~/.config/landsat-record/`, readable only by you;
- the upload is deleted;
- the key is checked for write access, without writing anything.

The key's contents are never printed. Skip this if you won't publish.

In [ ]:
!landsat-record login ~/ldeo-glaciology-key.json --remove-original

## 2. The whole thing in one line

This is a test run: about a dozen scenes covering every sensor, written to separate `_test` folders, ending with a
projection of the full run.

- **The full record:** drop `--test`.
- **Publishing:** add `--publish`.
- **Long runs:** better in a terminal, where you get live progress bars. Every run can be stopped and started again.

The outline is needed on a site's first run only, as one of:

- `--geojson FILE`
- `--bbox W S E N` in lon/lat, e.g. `--bbox -68.6 -71.5 -67.9 -71.2`
- `--bbox-3031 XMIN YMIN XMAX YMAX` in metres

In [ ]:
!landsat-record run flask_region --geojson ../sites/flask_region.geojson --title "Flask Glacier" --test

## 3. The same steps from Python

A `Site` is a name for the folders, a title for the viewer and an outline. A `LandsatRecord` knows where the site's
files go on scratch and runs each step. `test=True` uses the test sample and the `_test` folders, as above.

In [ ]:
from landsat_record import LandsatRecord, Site

site = Site.from_geojson("flask_region", "../sites/flask_region.geojson", title="Flask Glacier")
# or Site.from_bbox("fossil_bluff", -68.6, -71.5, -67.9, -71.2, title="Fossil Bluff")
# or Site.from_bbox_3031("my_site", xmin, ymin, xmax, ymax, title="My site")

record = LandsatRecord(site, test=True)
record

**Search** the USGS catalogue. There is no cloud, date or sun-angle filter: the aim is the complete record.

In [ ]:
record.search()
print(record.site.describe())
print(record.out_root)

**Crop** every band to the site, with no resampling. A run can be interrupted and restarted: finished bands
are skipped and failed ones are tried again. The bands table says what happened to each band.

In [ ]:
record.check_access()  # fails now, rather than hours in, if the USGS bucket can't be read
bands = record.crop()
bands.groupby(["status", "band"]).size().unstack("status", fill_value=0)

After a test run, project the full run's time, size and request cost:

In [ ]:
record.project_full_run()

A quick look at each scene, using the finest band with data. The red line is the site outline. Check that
the site falls inside it, that SLC-off stripes show on later ETM+ scenes, and how night scenes behave.

In [ ]:
record.quicklook();

**COGs, browse images and the viewer.** Both go into a separate `_cog` folder, which is what gets published:

- each conversion is checked against its source before upload;
- the browse images are 8-bit WebPs, stretched per scene.

In [ ]:
cog_bands, failures = record.cogs()
index = record.browse()

## 4. Look at the results

In [ ]:
scenes = record.scenes()
scenes[scenes.scene_status != "not_run"][["scene_id", "datetime", "prefix", "cloud_cover", "sun_elevation", "scene_status"]]

Do all the scenes share one pixel lattice at each resolution? One row per resolution means a stack needs no
resampling. Several rows mean scenes are offset by sub-pixel amounts.

In [ ]:
bands = record.bands()
tifs = bands[(bands.status == "ok") & (bands.band != "MTL.json")]
(
    tifs.assign(x_phase=(tifs.x_origin % tifs.res).round(3), y_phase=(tifs.y_origin % tifs.res).round(3))
    .groupby(["res", "x_phase", "y_phase"])
    .agg(n_bands=("band", "size"), n_scenes=("scene_id", "nunique"))
)

Open one COG from scratch. `bands(cog=True)` points at the COGs:

In [ ]:
import matplotlib.pyplot as plt
from rasterio.io import MemoryFile
from rasterio.plot import show

cogs = record.bands(cog=True)
row = cogs[(cogs.band == "red") & (cogs.status == "ok")].iloc[-1]
with MemoryFile(record.fs.cat(row.key)) as mem, mem.open() as src:
    print(row.scene_id, src.crs, src.res, src.shape)
    show(src, cmap="gray")

## 5. Publish

This copies the `_cog` folder to `gs://ldeo-glaciology/landsat/<site>`, or `<site>_test` for a test run. It then
returns the viewer's address. Only new files are copied, apart from the index and tables.

**Check before publishing:**
- Everything published is readable by anyone. Landsat data is public domain.
- Downloads are billed to the bucket's owner.

In [ ]:
PUBLISH = False

if PUBLISH:
    print(record.publish())

When only the viewer page has changed, for example after upgrading this package, this replaces just
`index.html`. People may see the old page for up to an hour, because of the bucket's caching.

In [ ]:
UPDATE_VIEWER = False

if UPDATE_VIEWER:
    print(record.update_viewer())

## 6. The full record

Once the test looks right, run the whole record, in a terminal for long runs:

```bash
landsat-record run flask_region --publish
```

From Python, make the record with `LandsatRecord(site)`, with `test=False`, and run the same steps.